##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma 2、Gemini 和 RouteLLM 入門

[Gemma](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開源語言模式。 Gemma 模型採用與創建 Gemini 模型相同的研究和技術構建而成，是文本到文本、僅限解碼器的大語言模型 (LLM)，提供英語版本，具有開放權重、預訓練變體和指令調整變體。
Gemma 模型非常適合各種文本生成任務，包括問答、總結和推論。它們的尺寸相對較小，因此可以將其部署在筆記型電腦、桌上型電腦或雲端基礎設施等資源有限的環境中，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
[Gemini](https://blog.google/technology/ai/google-gemini-ai/)是 Google AI 開發的大型語言模型。它是一個多模式模型，這意味著它可以處理和生成文字、程式碼、圖像和音訊。 Gemini 被認為是最先進的語言模型之一，它已被證明在各種任務上優於其他模型，例如問答、摘要和翻譯。
[RouteLLM](https://github.com/lm-sys/RouteLLM) 是 framework，可根據查詢的複雜度將查詢路由到最合適的模型，從而幫助您優化 LLM 使用。它可以在不犧牲性能的情況下顯著降低成本。您可以輕鬆地將其整合到現有應用程式中並嘗試不同的路由策略。
在此notebook 中，您將了解如何在Google Colab 環境中使用**RouteLLM** 在Gemini 和Gemma 2 模型之間進行路由。您將安裝必要的軟體包、設定模型並執行範例prompt。
<table align="left">
<td> <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_2]Using_Gemini_and_Gemma_with_RouteLLM.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />執行Google Colab</a>
</td>
</table>

## 設定

### 選擇 Colab runtime
要完成本教學，您必須擁有 Colab runtime 以及足夠的資源來執行 Gemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### 設定 Hugging Face 和Gemini

**在深入學習本教學之前，讓我們先設定 Hugging Face 和 Gemma：**
#### Hugging Face設置

1. **Hugging Face 帳戶：** 如果您還沒有帳戶，您可以點選[此處](https://huggingface.co/join) 建立免費帳戶。

2. **Hugging Face token：** 透過點選[此處](https://huggingface.co/settings/tokens) 產生Hugging Face 存取權限（最好是`write` 權限）token。在本教學的後面部分，您將需要這個token。

#### Gemini設置

1. **Gemini token：** 若要使用Gemini API，您需要API 金鑰。您只需在 [Google AI Studio](https://aistudio.google.com/app/apikey) 中點選幾下即可建立金鑰。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的 HF token 和 Gemini token

將您的 Hugging Face token 和 Gemini token 新增至 Colab Secrets manager 以安全地儲存它。
1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. 建立一個新的secret，名稱為`HF_TOKEN`。
3. 將 HF token 金鑰複製/貼上到 `HF_TOKEN` 的值輸入框中。
4. 切換左側的按鈕以允許notebook 存取secret。
5. 建立一個新的secret，名稱為`GOOGLE_API_KEY`。
6. 將Gemini token 金鑰複製/貼上到`GOOGLE_API_KEY` 的值輸入框中。
7. 切換左側的按鈕以允許notebook 存取secret。

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")

目前，RouteLLM 在啟動之前會檢查 `OPENAI_API_KEY`。您可以將 `OPENAI_API_KEY` 設定為虛擬值作為臨時解決方法。 RouteLLM 的合作者目前正在努力解決這個[問題](https://github.com/lm-sys/RouteLLM/issues/19)。

In [ ]:
os.environ["OPENAI_API_KEY"] = "dummy"

### 安裝依賴項

您需要安裝一些 Python 軟體包以及 RouteLLM 和 Gemini API 的依賴項。
執行以下cell來安裝或升級它：

In [ ]:
# Install RouteLLM package.
! pip install "routellm[serve,eval]"

# Install the Python SDK for the Gemini API, `google-generativeai`.
! pip install -q google-generativeai

### 使用Ollama 設定Gemma2

要將 Gemma 2 等本地模型與 RouteLLM 一起使用，您需要 [Ollama](https://ollama.com/)。
Ollama 是一個開源 framework，用於建立和執行大型語言模型 (LLM)。它的設計靈活且可定制，允許開發人員訓練和部署他們的模型或微調現有模型。對於想要嘗試 LLM 或在不依賴專有平台的情況下建立自訂模型的人來說，Ollama 是一個受歡迎的選擇。

#### Install Ollama

In [ ]:
!curl https://ollama.ai/install.sh | sh

#### 啟動 Ollama 作為後台子進程

In [ ]:
import subprocess
import time

process = subprocess.Popen("ollama serve", shell=True)
time.sleep(5)

#### 在 Ollama 中執行 Gemma 2 作為後台子進程

In [ ]:
process = subprocess.Popen("ollama run gemma2", shell=True)
time.sleep(5)

#### 檢查Ollama是否正在執行

執行以下命令查看Ollama是否已啟動並正在執行。當下列指令的輸出顯示「**Ollama is running**」時，繼續執行下一個cell。

In [ ]:
!curl localhost:11434

## RouteLLM 如何運作？


RouteLLM 是一個系統framework，用於基於偏好資料的 LLM 路由。 RouteLLM 的路由設定有兩個模型：一個較弱但成本較低的模型和一個較強但成本較高的模型。
當 prompt 傳送到 RouteLLM 時，底層路由方法會將 prompt 路由至 inference 的強模型或弱模型。 RouteLLM 提供四種不同的路由方法：
1. **相似性加權 (SW) 排名**：相似性加權 (SW) 排名路由器，根據相似性執行「加權 Elo 計算」。

2. **矩陣分解**：矩陣分解模型，用於學習模型回答 prompt 的評分函數。

3. **BERT 分類器**：BERT 分類器，用於預測哪個模型可以提供更好的反應。

4. **因果 LLM 分類器**：因果 LLM 分類器，用於預測哪個模型可以提供更好的回應。

從 [lmsys 的 blog](https://lmsys.org/blog/2024-07-01-routellm/) 中了解有關這些路由方法及其效能統計資料的更多資訊。
您也可以參考研究論文 [RouteLLM: Learning to Route LLMs with Preference
資料](https://arxiv.org/abs/2406.18665) 由創作者發佈。

## 使用`routellm` library 在Gemini 和Gemma 2 之間路由


您可以使用 `routellm` library 在 Python 程式碼中使用 **RouteLLM**。
您將透過指定路由器、strong_model 和 weak_model 從`routellm` library 初始化`Controller`。以下是 `Controller` 的每個參數的作用：
- `routers`：路由器的名稱。對於這個notebook，您將選擇`bert`，因為在 colab 環境中執行`bert`更容易。
- `strong_model`：更強大、更昂貴的型號。對於本範例，您將使用 **Gemini Pro**。
- `weak_model`：較弱但較便宜的型號。您可以將本地 **Gemma 2** 模型放在這裡。

In [ ]:
from routellm.controller import Controller

client = Controller(
  routers=["bert"],  # Use `bert` router
  strong_model="gemini/gemini-pro",
  weak_model="ollama_chat/gemma2"
)

**閾值校準**對於平衡 LLM 路由的成本和品質至關重要。最佳閾值取決於您的路由器和傳入查詢。使用範例和所需的路由百分比來校準您的查詢。 RouteLLM 支援基於公共[Chatbot Arena dataset](https://huggingface.co/datasets/lmsys/lmsys-arena-human-preference-55k)的預設校準。
在此範例中，您將校準 `bert` 的閾值，以便 30% 的呼叫被路由到更強的模型。要計算此閾值，您可以執行 `routellm.calibrate_threshold` 命令並提供以下值。
`--routers`：伯特
`--strong-model-pct`：0.3

In [ ]:
!python -m routellm.calibrate_threshold --task calibrate --routers bert --strong-model-pct 0.3 --config config.example.yaml

`client.chat.completions.create` 函數可讓您prompt 您的 RouteLLM 設定。您可以在 `client.chat.completions.create` 函數的 `model` 參數中指定在上一個步驟中獲得的閾值。如果路由器是 `bert` 且閾值是 **0.46514**，則將 `router-bert-0.46514` 傳遞給 `model` 參數。您可以將聊天訊息傳遞給`messages` 參數。

In [ ]:
response = client.chat.completions.create(
  # This tells RouteLLM to use the bert router with a cost threshold of 0.46514
  model="router-bert-0.46514",
  messages=[
    {"role": "user", "content": "Hello!"}
  ]
)

print("Selected model: {model}\n".format(model=response.model))
print("Response: {model_response}".format(
    model_response=response.choices[0].message.content))

要了解有關 **RouteLLM** 的 Python 用法的更多信息，請訪問 [RouteLLM 的 GitHub 頁面](https://github.com/lm-sys/RouteLLM)。

## 結論

恭喜！您已在 Google Colab 環境中使用 **RouteLLM** 在 Gemini 和 Gemma 2 模型之間成功進行路由。